# Collector + Maintainer Pilot — v4

This notebook instantiates the three model-assisted stages used in the Nepal proof of concept:

1. **Collection:** retrieve source-backed evidence from a trusted registry and the open web while preserving provenance.
2. **Filing:** classify each evidence record against the NGO bow-tie taxonomy without browsing.
3. **Article maintenance:** update five citation-bound threat-line articles across the three revision dates.

## Four-state editorial policy

The maintained articles use four statuses:

| Status | Operational meaning |
|---|---|
| `unassessed` | The available evidence is absent or insufficient to support a substantive judgment. |
| `low_risk` | Affirmative, supported evidence indicates that no currently effective restriction or loss is established for the line. |
| `high_risk` | Supported restrictive evidence shows meaningful exposure, deterioration, or an advanced proposal, but direct operational loss is not established. |
| `materialised` | Supported restrictive evidence establishes a currently effective restriction, loss, or condition with direct relevance to operational viability. |

`low_risk` is never the default for an empty category. A line with no supported restrictive or permissive evidence remains `unassessed`. The classifier distinguishes affirmative permissive evidence from mere silence, and the article-maintenance stage receives a mechanically derived cumulative coverage summary that is validated against its proposed status.

## Running the notebook

Set `EVIDENCE_MODE` in Section 2:

- `"artifact"` loads the collector artifact distributed with the supplementary material, supporting reproducible downstream filing and maintenance.
- `"live"` performs a new web-enabled collection and writes a new collector artifact.

The notebook stores no API key and contains no saved cell outputs. Generated filing and article-maintenance artifacts use `v4` filenames.


## 1. Setup

Install the required packages in the active kernel and load `ANTHROPIC_API_KEY` from a local `.env` file. Do not place credentials directly in the notebook.

1. Copy `.env.example` to `.env`.
2. Set `ANTHROPIC_API_KEY=sk-ant-...` in `.env` with your real key (`.env` is already gitignored).

In [ ]:
# Install dependencies in the active notebook environment.
%pip install -q anthropic pandas python-dotenv

In [ ]:
import calendar
import datetime as dt
import hashlib
import json
import os
import re
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from a local .env file, if present

assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY is not set (add it to .env)"
print("API key found")

## 2. Configuration


In [ ]:
MODEL = "claude-sonnet-5"

# Evidence source for this notebook:
# - "artifact": load the collector artifact distributed with the supplement.
# - "live": perform a new web-enabled collection.
EVIDENCE_MODE = "live"

COLLECTOR_ARTIFACT_PATH = Path("collector_run.json")
LIVE_COLLECTOR_OUT_PATH = Path("collector_run_v4.json")
CLASSIFICATION_OUT_PATH = Path("classification_run_v4.json")
CLASSIFIED_CSV_PATH = Path("classified_evidence_v4.csv")
ARTICLE_REVISIONS_OUT_PATH = Path("article_revisions_v4.json")

MAX_AGENT_TURNS = 8
MAX_CONTINUATION_TURNS = 4
COLLECTOR_MAX_TOKENS = 16000
CLASSIFICATION_BATCH_SIZE = 8
MAX_MAINTAINER_REPAIR_ATTEMPTS = 1

if EVIDENCE_MODE not in {"artifact", "live"}:
    raise ValueError("EVIDENCE_MODE must be either 'artifact' or 'live'.")


## 3. Shared JSON and provenance helpers

In [ ]:
def utc_now_iso():
    return dt.datetime.now(dt.timezone.utc).isoformat()


def parse_json_text(text):
    """Parse JSON even when the model accidentally wraps it in a code fence."""
    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        starts = [p for p in (cleaned.find("["), cleaned.find("{")) if p >= 0]
        if not starts:
            raise
        start = min(starts)
        end = max(cleaned.rfind("]"), cleaned.rfind("}"))
        if end <= start:
            raise
        return json.loads(cleaned[start:end + 1])


def salvage_truncated_array(text):
    """Best-effort recovery of a JSON array: keep every top-level object that
    parses on its own, dropping only the one(s) that are malformed or cut off
    -- whether that is a truncated tail, or a single corrupted record
    somewhere in the middle of the array.

    Returns a list, or raises if nothing salvageable. Only used after normal
    parsing fails, and the salvage is logged so the loss is visible.
    """
    start = text.find("[")
    if start < 0:
        raise ValueError("No JSON array found to salvage.")

    spans, depth, in_str, esc, obj_start = [], 0, False, False, None
    for i, ch in enumerate(text[start:], start):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"':
            in_str = True
        elif ch in "[{":
            if depth == 1 and ch == "{" and obj_start is None:
                obj_start = i
            depth += 1
        elif ch in "]}":
            depth -= 1
            if depth == 1 and ch == "}" and obj_start is not None:
                spans.append((obj_start, i))
                obj_start = None

    salvaged = []
    for span_start, span_end in spans:
        try:
            salvaged.append(json.loads(text[span_start:span_end + 1]))
        except json.JSONDecodeError:
            continue  # drop just this one bad record; keep the rest

    if not salvaged:
        raise ValueError("No complete object found in truncated array.")
    return salvaged


def domain_from_url(url):
    try:
        return urlparse(url).netloc.lower().removeprefix("www.")
    except Exception:
        return ""


def stable_evidence_id(item):
    seed = "|".join([
        str(item.get("url", "")),
        str(item.get("event_date_iso", item.get("date_iso", ""))),
        str(item.get("snippet", "")),
    ])
    return "ev_" + hashlib.sha256(seed.encode("utf-8")).hexdigest()[:12]

## 4. Source registry

The registry is explicit so that collection scope can be reproduced and audited.


In [ ]:
REGISTRY_DOMAINS = [
    "icnl.org",
    "monitor.civicus.org",
]

## 5. Collector instructions

The collector retrieves both restrictive and risk-reducing evidence. It preserves provenance but does not classify records or assign article statuses.


In [ ]:
COLLECTOR_PROMPT = """You are the COLLECTOR component of a civic-space
monitoring pilot for an NGO operating in Nepal.

Retrieve concrete source-backed items that may describe the NGO operating
environment, including laws, bills, policies, official decisions, funding
changes, court or regulatory actions, public statements, protests, elections,
and other events potentially relevant to civil society.

Retrieve both risk-increasing and risk-reducing evidence. Risk-reducing evidence
includes legal protections, appeal rights, withdrawals, repeals, suspensions,
reversals, court decisions restoring a protection, and authoritative statements
that a relevant restriction is not in force or does not apply. Do not treat the
absence of search results, or a source's silence about a restriction, as evidence
that risk is low. Return only affirmative, source-backed records.

Separation of roles:
- You retrieve candidate evidence and preserve provenance.
- You DO NOT assign a bow-tie threat category.
- You DO NOT label an item restrictive, permissive, or contextual.
- You DO NOT assign a risk status or recommend an action.
- Keep descriptions factual and close to what the source states.

Tools:
- Consult the trusted registry first: ICNL Civic Freedom Monitor and CIVICUS
  Monitor.
- Use unrestricted web search to supplement the registry with official sources
  and reputable reporting, especially for recent events the monitors may lag on.
- Before including an item, open its page with web_fetch and take the
  verbatim snippet from the fetched page itself. Do not rely on a search
  result preview alone -- a search snippet is not sufficient grounding.

For each concrete item, return:
- title: source or document title;
- description: one short factual description of the item;
- event_date_text: the item's date or year as written or clearly established;
- event_date_iso: normalized event date as YYYY-MM-DD, YYYY-MM, or YYYY; null
  when the source does not establish one. This is the date of the law/event,
  not merely the page's last-updated date;
- source_publication_date: publication/update date of the retrieved page when
  available, otherwise null;
- url: exact fetched URL;
- snippet: a short verbatim passage that directly supports the description;
- document_type: one of law, bill, policy, official_statement, court_filing,
  monitor_report, news_report, funding_notice, election_result, or other.

Do not include an item unless its description is directly supported by the
verbatim snippet. If one source supports several distinct events, return
separate records. Prefer primary or monitor sources and avoid duplicates.

Output ONLY a JSON array of objects with exactly those keys."""


## 6. Web tool definitions

The server-tool version strings are explicit for reproducibility. If the provider retires a version, update only this cell to the current compatible specification and record the change with the released artifact.


In [ ]:
TOOLS = [
    {
        "type": "web_search_20250305",
        "name": "web_search",
        "max_uses": 10,
    },
    {
        "type": "web_fetch_20250910",
        "name": "web_fetch",
        "max_uses": 14,
        "max_content_tokens": 3000,
        # No allowed_domains restriction: web_fetch can only open URLs that
        # web_search already surfaced in this conversation, so this does not
        # let the model wander the open web -- it just lets it open any of
        # the sources search already found, not only the 2 registry sites.
        # max_content_tokens caps each fetched page so a single long page
        # can't eat the whole per-turn token budget or blow up cost.
    },
]

## 7. Worked-example recall targets

Used only for a descriptive recall check; never passed to the collector.

In [ ]:
WORKED_EXAMPLE_TARGETS = [
    "Social Welfare Act foreign funding approval / domestic banks",
    "2018 draft National Integrity Policy (withdrawn)",
    "2019 IT bill on online content",
    "2019 Home-Ministry NGO law in drafting",
    "2020 policy on INGO programmes",
    "2023 CIVICUS obstructed rating",
    "August 2025 draft NGO Registration Regulation and Management Bill",
    "Social Welfare Council dissolution",
    "September 2025 blocking of 26 social-media platforms",
    "2025 USAID closure or development funding withdrawal",
    "Gen Z protests and crackdown in 2025",
    "platform block withdrawn in 2025",
    "March 2026 election or reform government",
    "June 2026 National Child Rights Council dissolution proposal",
]

## 8. Collector runner

The runner handles tool turns and output truncation explicitly. If a JSON array remains incomplete after continuation attempts, it retains only complete records and logs that salvage event instead of silently passing an empty evidence set downstream.


In [ ]:
def normalise_collector_items(items, retrieved_at):
    normalised = []
    for raw in items:
        item = dict(raw)

        # Normalize optional aliases and fields returned by the collector.
        item.setdefault("title", "")
        item.setdefault("event_date_text", item.pop("date", None))
        item.setdefault("event_date_iso", item.pop("date_iso", None))
        item.setdefault("source_publication_date", None)
        item.setdefault("document_type", "other")

        item["url"] = str(item.get("url", "")).strip()
        item["source_domain"] = domain_from_url(item["url"])
        item["source_tier"] = (
            "registry"
            if any(
                item["source_domain"] == d or item["source_domain"].endswith("." + d)
                for d in REGISTRY_DOMAINS
            )
            else "supplementary"
        )
        item["retrieved_at"] = retrieved_at
        item["evidence_id"] = stable_evidence_id(item)
        normalised.append(item)

    deduplicated, seen = [], set()
    for item in normalised:
        if item["evidence_id"] not in seen:
            deduplicated.append(item)
            seen.add(item["evidence_id"])
    return deduplicated


def run_collector(model):
    client = Anthropic()
    retrieved_at = utc_now_iso()
    log = {
        "stage": "collector",
        "model": model,
        "timestamp": retrieved_at,
        "registry": REGISTRY_DOMAINS,
        "fetched_urls": [],
        "searches": [],
        "raw_block_types": [],
        "stop_reasons": [],
        "extracted": None,
    }

    messages = [{
        "role": "user",
        "content": (
            "Collect a retrospective candidate-evidence set for Nepal covering "
            "the period needed to reconstruct the baseline at 31 December 2024, "
            "the developments during 2025, and developments through June 2026."
        ),
    }]

    final_text_parts = []          # accumulates text across continuation turns
    continuations_used = 0

    for _ in range(MAX_AGENT_TURNS + MAX_CONTINUATION_TURNS):
        resp = client.messages.create(
            model=model,
            max_tokens=COLLECTOR_MAX_TOKENS,
            system=COLLECTOR_PROMPT,
            tools=TOOLS,
            messages=messages,
        )
        log["stop_reasons"].append(resp.stop_reason)

        for block in resp.content:
            btype = getattr(block, "type", None)
            log["raw_block_types"].append(btype)

            if btype == "server_tool_use":
                name = getattr(block, "name", "")
                block_input = getattr(block, "input", {}) or {}
                if name == "web_search":
                    log["searches"].append(block_input.get("query", ""))

            if btype == "web_fetch_tool_result":
                url = getattr(block, "url", None)
                if url:
                    log["fetched_urls"].append(url)

            if btype == "web_search_tool_result":
                for result in getattr(block, "content", []) or []:
                    url = getattr(result, "url", None)
                    if url:
                        log["fetched_urls"].append(url)

        turn_text = "".join(
            getattr(block, "text", "")
            for block in resp.content
            if getattr(block, "type", None) == "text"
        )

        if resp.stop_reason in {"tool_use", "pause_turn"}:
            messages.append({"role": "assistant", "content": resp.content})
            messages.append({"role": "user",
                             "content": "Continue and finish the JSON output."})
            continue

        # Treat truncated output as a continuation case, not a final answer.
        if resp.stop_reason == "max_tokens":
            continuations_used += 1
            if continuations_used > MAX_CONTINUATION_TURNS:
                log["parse_error"] = "Exceeded MAX_CONTINUATION_TURNS while continuing truncated output."
                log["extracted_raw_text"] = "".join(final_text_parts) + turn_text
                return log
            final_text_parts.append(turn_text)
            messages.append({"role": "assistant", "content": resp.content})
            messages.append({"role": "user", "content": (
                "Your JSON output was cut off by the length limit. Continue the "
                "JSON array EXACTLY where it stopped. Output only the remaining "
                "characters, with no preamble, no repetition of earlier records, "
                "and no code fences."
            )})
            continue

        # Assemble the full text after the final turn.
        final_text_parts.append(turn_text)
        final_text = "".join(final_text_parts).strip()

        try:
            parsed = parse_json_text(final_text)
            if not isinstance(parsed, list):
                raise ValueError("Collector output must be a JSON array.")
            log["extracted"] = normalise_collector_items(parsed, retrieved_at)
        except Exception as exc:
            # Last resort: keep the complete records from a broken array.
            try:
                salvaged = salvage_truncated_array(final_text)
                log["extracted"] = normalise_collector_items(salvaged, retrieved_at)
                log["salvage_note"] = (
                    f"Primary parse failed ({exc!r}); salvaged "
                    f"{len(salvaged)} complete records from truncated array."
                )
            except Exception:
                log["parse_error"] = repr(exc)
                log["extracted_raw_text"] = final_text
        return log

    raise RuntimeError(
        f"Collector did not finish after {MAX_AGENT_TURNS + MAX_CONTINUATION_TURNS} "
        "allowed turns. Increase the turn limits or inspect the API response."
    )


## 9. Obtain the evidence records

In `artifact` mode, the notebook reads the collector artifact supplied with the reproducibility package. In `live` mode, it performs collection and writes a new artifact. Both modes feed the same normalization, validation, filing, and maintenance code.


In [ ]:
if EVIDENCE_MODE == "live":
    collector_log = run_collector(MODEL)
    with LIVE_COLLECTOR_OUT_PATH.open("w", encoding="utf-8") as f:
        json.dump(collector_log, f, indent=2, ensure_ascii=False, default=str)
    active_collector_path = LIVE_COLLECTOR_OUT_PATH
    print(f"Live collector artifact saved to {active_collector_path}")
    print(f"Stop reasons observed: {collector_log['stop_reasons']}")
    if collector_log.get("salvage_note"):
        print("⚠️ ", collector_log["salvage_note"])
else:
    if not COLLECTOR_ARTIFACT_PATH.exists():
        raise FileNotFoundError(
            f"{COLLECTOR_ARTIFACT_PATH} was not found. Place the collector "
            "artifact from the supplementary package beside this notebook, "
            "or set EVIDENCE_MODE = 'live' in Section 2."
        )
    with COLLECTOR_ARTIFACT_PATH.open(encoding="utf-8") as f:
        collector_log = json.load(f)
    active_collector_path = COLLECTOR_ARTIFACT_PATH
    print(
        f"Loaded collector artifact from {active_collector_path} "
        f"(collection timestamp: {collector_log.get('timestamp')})"
    )

# Do not let an empty or failed collection flow downstream.
if collector_log.get("parse_error"):
    raise RuntimeError(
        "Collector JSON could not be parsed: "
        f"{collector_log['parse_error']}\n"
        "Inspect collector_log['extracted_raw_text'] before continuing."
    )

items = collector_log.get("extracted") or []
if not items:
    raise RuntimeError(
        "The collector artifact contains 0 evidence records. Downstream stages "
        "would produce empty classifications and article revisions. Inspect "
        f"{active_collector_path}, including stop_reasons, searches, and "
        "fetched_urls."
    )

print(f"Evidence records: {len(items)} ✔")


In [ ]:
from collections import Counter

# Sanity check: confirm the collector actually opened pages (web_fetch),
# not just search snippets, on this run.
block_counts = Counter(collector_log.get("raw_block_types", []))
pages_opened = block_counts.get("web_fetch_tool_result", 0)
searches_run = len(collector_log.get("searches", []))

print(f"Searches run: {searches_run}")
print(f"Pages actually opened (web_fetch): {pages_opened}")

if pages_opened == 0:
    print(
        "\n⚠️  No pages were opened this run -- the collector relied only on "
        "search snippets. Check that you re-ran the prompt/tools cells with the "
        "web_fetch changes before re-running the collector."
    )
else:
    print(
        f"\n✔ Confirmed: the collector opened {pages_opened} page(s) with web_fetch "
        "this run, not just search-result snippets."
    )

In [ ]:
print(f"Model:             {MODEL}")
print(f"Searches issued:   {len(collector_log.get('searches', []))}")
print(f"Unique URLs seen:  {len(set(collector_log.get('fetched_urls', [])))}")
print(f"Evidence records:  {len(items)}")

for item in items:
    print(
        f"\n{item['evidence_id']} | {item.get('event_date_text')} | "
        f"{item.get('description', '')}"
    )
    print(f"  {item.get('url', '')}")
    print(f"  snippet: {str(item.get('snippet', ''))[:180]}")

## 10. Descriptive recall check

A deliberately crude screen; manual review remains the reference.

In [ ]:
def target_keywords(target):
    stop = {"with", "from", "into", "that", "this", "current", "report"}
    return [
        token
        for token in re.findall(r"[a-z0-9]+", target.lower())
        if len(token) > 3 and token not in stop
    ]


corpus = " ".join(
    " ".join([
        str(item.get("title", "")),
        str(item.get("description", "")),
        str(item.get("snippet", "")),
        str(item.get("event_date_text", "")),
    ])
    for item in items
).lower()

hits = 0
for target in WORKED_EXAMPLE_TARGETS:
    kws = target_keywords(target)
    overlap = sum(keyword in corpus for keyword in kws)
    found = overlap >= max(1, (len(kws) + 1) // 2)
    hits += int(found)
    print(f"[{'x' if found else ' '}] {target} ({overlap}/{len(kws)} keywords)")

print(f"\nNaive target recall: {hits}/{len(WORKED_EXAMPLE_TARGETS)}")

# Part II — Maintainer filing/classification pilot

This stage uses **only the supplied evidence records**. It does not browse and must not introduce facts from model memory.

## 11. Taxonomy and barriers


In [ ]:
THREAT_TAXONOMY = {
    "governance": [
        "unchecked_regulator_powers",
        "loss_of_appeal_rights",
        "restricted_freedoms",
        "unilateral_rule_making",
    ],
    "formation": [
        "forced_reregistration",
        "arbitrary_denial",
        "capital_membership_bars",
        "mandatory_state_partner",
    ],
    "operations": [
        "per_project_permits",
        "foreign_funding_activity_bans",
        "forced_coordination",
        "asset_surrender",
    ],
    "resources": [
        "foreign_funding_approval",
        "foreign_funding_caps_or_bans",
        "mandatory_state_banking",
        "punitive_taxation",
    ],
    "reputational_and_political": [
        "foreign_agent_labelling",
        "hostile_official_rhetoric",
        "coordinated_media_campaigns",
        "criminalising_association",
    ],
}

BARRIERS = {
    "governance": "appeal_and_judicial_review",
    "formation": "statutory_deadlines_and_bounded_refusal",
    "operations": "general_operating_licence_and_self_regulation",
    "resources": "diversified_funding_and_transparent_reporting",
    "reputational_and_political": "public_trust_and_proactive_transparency",
}

POLARITIES = ["restrictive", "permissive", "contextual"]
TAXONOMY_FITS = ["exact", "partial", "outside_current_subcategories"]
GROUNDING_STATUSES = ["supported", "partially_supported", "unsupported"]

## 12. Classification prompt and runner

The filing prompt treats restrictive and permissive evidence symmetrically. Crucially, `permissive` means affirmative evidence of protection, reversal, non-applicability, or the current absence of a restriction; it never means that the snippet simply failed to mention a restriction.


In [ ]:
CLASSIFIER_PROMPT = f"""You are the FILING stage of the MAINTAINER in an
LLM-assisted NGO bow-tie monitoring system.

You receive candidate evidence records produced by the Collector. Use ONLY the
fields supplied in those records. Do not browse, do not use outside knowledge,
and do not infer facts that are absent from the verbatim snippet.

Taxonomy:
{json.dumps(THREAT_TAXONOMY, indent=2)}

Category-level preventive barriers:
{json.dumps(BARRIERS, indent=2)}

Polarity meanings:
- restrictive: affirmative evidence of a threat, restriction, deterioration,
  loss, or proposal that could increase exposure;
- permissive: affirmative evidence of a protection, appeal right, repeal,
  withdrawal, lapse, suspension, reversal, non-applicability, or authoritative
  confirmation that a relevant restriction is not currently in force;
- contextual: evidence useful for interpretation but insufficient by itself to
  establish either a restrictive or permissive condition.

A record is NOT permissive merely because it does not mention a restriction.
Silence, missing information, failed retrieval, or absence of a restrictive
statement must not be converted into evidence of low risk.

For EACH evidence record:
1. Check whether the description is grounded by the snippet.
2. Decide whether the snippet contains evidence relevant to NGO operational
   viability. General political news may be contextual, but it should not be
   forced into a threat category without a clear connection.
3. Treat evidence of improvement as relevant when it affirmatively documents a
   withdrawal, repeal, lapse, suspension, reversal, legal protection, or other
   change affecting a documented restriction or proposal. File broad political
   transitions as contextual unless the snippet directly establishes their
   effect on a specific measure.
4. Return zero, one, or several assignments. Multiple assignments are allowed
   only when the same snippet independently supports each one.
5. For each assignment choose:
   - category: exactly one taxonomy category;
   - subcategory: one listed subcategory, or "other";
   - taxonomy_fit: exact, partial, or outside_current_subcategories;
   - polarity: restrictive, permissive, or contextual;
   - affected_barrier: the category-level barrier shown above, or
     "none_identified";
   - rationale: a concise explanation based only on the snippet;
   - supporting_text: a short verbatim substring copied from the supplied
     snippet;
   - confidence: a number from 0 to 1.
6. Set requires_human_review to true when grounding is partial/unsupported,
   confidence is below 0.70, the taxonomy fit is outside the current
   subcategories, or the evidence could reasonably be interpreted in more than
   one way.
7. If irrelevant, return relevant=false and an empty assignments list.
8. Do NOT assign unassessed, low risk, high risk, or materialised. Article-state
   maintenance is a later operation.

Output ONLY one JSON object:
{{
  "classifications": [
    {{
      "evidence_id": "exact input evidence_id",
      "grounding_status": "supported|partially_supported|unsupported",
      "grounding_note": "brief explanation",
      "relevant": true,
      "assignments": [
        {{
          "category": "governance|formation|operations|resources|reputational_and_political",
          "subcategory": "allowed subcategory or other",
          "taxonomy_fit": "exact|partial|outside_current_subcategories",
          "polarity": "restrictive|permissive|contextual",
          "affected_barrier": "allowed barrier or none_identified",
          "rationale": "brief evidence-bounded rationale",
          "supporting_text": "verbatim substring from the input snippet",
          "confidence": 0.0
        }}
      ],
      "requires_human_review": false,
      "review_note": ""
    }}
  ]
}}
"""


def compact_evidence(item):
    return {
        "evidence_id": item["evidence_id"],
        "title": item.get("title"),
        "description": item.get("description"),
        "event_date_text": item.get("event_date_text"),
        "event_date_iso": item.get("event_date_iso"),
        "url": item.get("url"),
        "snippet": item.get("snippet"),
        "document_type": item.get("document_type"),
        "source_tier": item.get("source_tier"),
    }


def run_classifier(evidence_items, model, batch_size=CLASSIFICATION_BATCH_SIZE):
    if not evidence_items:
        raise ValueError("run_classifier received 0 evidence items.")

    client = Anthropic()
    run = {
        "stage": "maintainer_filing",
        "model": model,
        "timestamp": utc_now_iso(),
        "taxonomy": THREAT_TAXONOMY,
        "barriers": BARRIERS,
        "classifications": [],
        "batches": [],
    }

    for start in range(0, len(evidence_items), batch_size):
        batch = evidence_items[start:start + batch_size]
        payload = [compact_evidence(item) for item in batch]

        resp = client.messages.create(
            model=model,
            max_tokens=8000,
            system=CLASSIFIER_PROMPT,
            messages=[{
                "role": "user",
                "content": (
                    "Classify every evidence record below. Return exactly one "
                    "classification per evidence_id.\n\n"
                    + json.dumps(payload, ensure_ascii=False, indent=2)
                ),
            }],
        )

        # Treat truncated classifier output as an error, not something to parse.
        if resp.stop_reason == "max_tokens":
            raise RuntimeError(
                f"Classifier batch starting at index {start} hit max_tokens. "
                "Lower CLASSIFICATION_BATCH_SIZE or raise max_tokens."
            )

        text = "".join(
            getattr(block, "text", "")
            for block in resp.content
            if getattr(block, "type", None) == "text"
        ).strip()
        parsed = parse_json_text(text)
        classifications = (
            parsed.get("classifications", [])
            if isinstance(parsed, dict)
            else parsed
        )
        if not isinstance(classifications, list):
            raise ValueError("Classifier output does not contain a classifications list.")

        run["classifications"].extend(classifications)
        run["batches"].append({
            "input_ids": [item["evidence_id"] for item in batch],
            "raw_text": text,
        })

    return run


## 13. Run classification and validate the output

In [ ]:
classification_run = run_classifier(items, MODEL)

with CLASSIFICATION_OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(classification_run, f, indent=2, ensure_ascii=False)

print(f"Classification run saved to {CLASSIFICATION_OUT_PATH}")

In [ ]:
def validate_classifications(evidence_items, classifications):
    evidence_by_id = {item["evidence_id"]: item for item in evidence_items}
    expected_ids = set(evidence_by_id)
    returned_ids = [row.get("evidence_id") for row in classifications]
    issues = []

    missing = expected_ids - set(returned_ids)
    duplicates = {eid for eid in returned_ids if returned_ids.count(eid) > 1}
    extra = set(returned_ids) - expected_ids

    if missing:
        issues.append(f"Missing classifications: {sorted(missing)}")
    if duplicates:
        issues.append(f"Duplicate classifications: {sorted(duplicates)}")
    if extra:
        issues.append(f"Unknown evidence IDs: {sorted(extra)}")

    for row in classifications:
        eid = row.get("evidence_id")
        source = evidence_by_id.get(eid)
        if source is None:
            continue

        if row.get("grounding_status") not in GROUNDING_STATUSES:
            issues.append(f"{eid}: invalid grounding_status")

        assignments = row.get("assignments", [])
        if row.get("relevant") is False and assignments:
            issues.append(f"{eid}: irrelevant item has non-empty assignments")

        snippet = str(source.get("snippet", "")).lower()
        for idx, assignment in enumerate(assignments):
            category = assignment.get("category")
            subcategory = assignment.get("subcategory")
            polarity = assignment.get("polarity")
            taxonomy_fit = assignment.get("taxonomy_fit")
            barrier = assignment.get("affected_barrier")
            supporting_text = str(assignment.get("supporting_text", "")).strip()

            if category not in THREAT_TAXONOMY:
                issues.append(f"{eid}[{idx}]: invalid category {category!r}")
                continue
            if subcategory not in THREAT_TAXONOMY[category] + ["other"]:
                issues.append(
                    f"{eid}[{idx}]: invalid subcategory {subcategory!r} "
                    f"for {category}"
                )
            if polarity not in POLARITIES:
                issues.append(f"{eid}[{idx}]: invalid polarity {polarity!r}")
            if taxonomy_fit not in TAXONOMY_FITS:
                issues.append(f"{eid}[{idx}]: invalid taxonomy_fit {taxonomy_fit!r}")
            if barrier not in set(BARRIERS.values()) | {"none_identified"}:
                issues.append(f"{eid}[{idx}]: invalid barrier {barrier!r}")
            if supporting_text and supporting_text.lower() not in snippet:
                issues.append(
                    f"{eid}[{idx}]: supporting_text is not a verbatim "
                    "substring of the collector snippet"
                )

    return issues


classifications = classification_run.get("classifications", [])
validation_issues = validate_classifications(items, classifications)

print(f"Evidence records:       {len(items)}")
print(f"Classifications:        {len(classifications)}")
print(f"Validation issues:      {len(validation_issues)}")
for issue in validation_issues:
    print(" -", issue)

# Refuse to continue with an empty classification result.
if not classifications:
    raise RuntimeError("Classifier returned 0 classifications; do not proceed to Part III.")


## 14. Inspect and export the classification table

In [ ]:
rows = []
evidence_by_id = {item["evidence_id"]: item for item in items}

for result in classifications:
    evidence = evidence_by_id.get(result.get("evidence_id"), {})
    assignments = result.get("assignments") or [None]

    for assignment in assignments:
        row = {
            "evidence_id": result.get("evidence_id"),
            "event_date": evidence.get("event_date_text"),
            "title": evidence.get("title"),
            "description": evidence.get("description"),
            "url": evidence.get("url"),
            "source_tier": evidence.get("source_tier"),
            "grounding_status": result.get("grounding_status"),
            "relevant": result.get("relevant"),
            "requires_human_review": result.get("requires_human_review"),
            "review_note": result.get("review_note"),
        }
        if assignment:
            row.update({
                "category": assignment.get("category"),
                "subcategory": assignment.get("subcategory"),
                "taxonomy_fit": assignment.get("taxonomy_fit"),
                "polarity": assignment.get("polarity"),
                "affected_barrier": assignment.get("affected_barrier"),
                "confidence": assignment.get("confidence"),
                "rationale": assignment.get("rationale"),
                "supporting_text": assignment.get("supporting_text"),
            })
        rows.append(row)

classification_df = pd.DataFrame(rows)
classification_df.to_csv(CLASSIFIED_CSV_PATH, index=False)

print(f"Classification table saved to {CLASSIFIED_CSV_PATH}")
display(classification_df)

## 15. Simple descriptive statistics

These are not accuracy metrics; they describe the run.

In [ ]:
if not classification_df.empty:
    print("Grounding status")
    print(classification_df["grounding_status"].value_counts(dropna=False), "\n")

    print("Category")
    print(classification_df["category"].value_counts(dropna=False), "\n")

    print("Polarity")
    print(classification_df["polarity"].value_counts(dropna=False), "\n")

    print("Human review")
    print(classification_df["requires_human_review"].value_counts(dropna=False))

# Part III — Sequential article maintenance

## 16. Four-state editorial policy and revision windows

The policy separates insufficient evidence from an affirmative low-risk assessment. `unassessed` is the required state when a category has no supported restrictive or permissive evidence.


In [ ]:
STATUS_DEFINITIONS = {
    "unassessed": (
        "The available evidence is absent or insufficient to support a "
        "substantive judgment for this line."
    ),
    "low_risk": (
        "Affirmative, supported evidence indicates that no currently effective "
        "restriction or loss is established for this line. Low risk must not be "
        "assigned solely because no restrictive evidence was retrieved."
    ),
    "high_risk": (
        "Supported restrictive evidence shows meaningful exposure, deterioration, "
        "or an advanced proposal, but direct operational loss is not established."
    ),
    "materialised": (
        "Supported restrictive evidence establishes a restriction, loss, or "
        "condition currently in effect with direct relevance to operational viability."
    ),
}

ALLOWED_STATUSES = set(STATUS_DEFINITIONS)

REVISION_WINDOWS = [
    {"revision_id": "baseline_2024_12_31", "label": "Baseline", "cutoff": "2024-12-31"},
    {"revision_id": "peak_2025_12_31", "label": "Peak", "cutoff": "2025-12-31"},
    {"revision_id": "divergence_2026_06_30", "label": "Divergence", "cutoff": "2026-06-30"},
]

# Manual evidence_id -> revision_id overrides for items with ambiguous dates.
REVISION_OVERRIDES = {}


def end_date_from_partial_iso(value):
    """Convert YYYY, YYYY-MM, or YYYY-MM-DD to an inclusive date."""
    if not value:
        return None
    value = str(value).strip()

    if re.fullmatch(r"\d{4}", value):
        return dt.date(int(value), 12, 31)

    if re.fullmatch(r"\d{4}-\d{2}", value):
        year, month = map(int, value.split("-"))
        return dt.date(year, month, calendar.monthrange(year, month)[1])

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", value):
        return dt.date.fromisoformat(value)

    return None


def evidence_revision_id(item):
    eid = item["evidence_id"]
    if eid in REVISION_OVERRIDES:
        return REVISION_OVERRIDES[eid]

    event_date = end_date_from_partial_iso(item.get("event_date_iso"))
    if event_date is None:
        # Conservative fallback: infer only the year from event_date_text.
        match = re.search(r"\b(19|20)\d{2}\b", str(item.get("event_date_text", "")))
        if match:
            event_date = dt.date(int(match.group(0)), 12, 31)

    if event_date is None:
        return None

    for window in REVISION_WINDOWS:
        if event_date <= dt.date.fromisoformat(window["cutoff"]):
            return window["revision_id"]
    return None


## 17. Pre-flight evidence-to-window diagnostic

Before the article-maintenance calls, this diagnostic shows exactly which classified records enter each revision window and identifies any record excluded because of an unusable date or missing classification.


In [ ]:
def revision_join_report(evidence_items, classification_rows):
    classification_by_id = {row.get("evidence_id"): row for row in classification_rows}
    per_window = {w["revision_id"]: [] for w in REVISION_WINDOWS}
    dropped = []

    for item in evidence_items:
        rid = evidence_revision_id(item)
        eid = item["evidence_id"]
        if rid is None:
            dropped.append((eid, "no usable event date — add to REVISION_OVERRIDES"))
        elif eid not in classification_by_id:
            dropped.append((eid, f"dated to {rid} but has no classification"))
        else:
            per_window[rid].append(eid)

    print("Evidence-to-revision join:")
    for window in REVISION_WINDOWS:
        ids = per_window[window["revision_id"]]
        print(f"  {window['revision_id']:<24} (≤ {window['cutoff']}): {len(ids)} items")
        for eid in ids:
            print(f"      {eid}")

    if dropped:
        print(f"\n⚠️  {len(dropped)} item(s) will NOT reach any revision:")
        for eid, reason in dropped:
            print(f"      {eid}: {reason}")

    total_joined = sum(len(v) for v in per_window.values())
    if total_joined == 0:
        raise RuntimeError(
            "The evidence↔window↔classification join produced ZERO items for "
            "every revision window. Check the dropped-item reasons above before "
            "running article maintenance."
        )
    return per_window


join_preview = revision_join_report(items, classifications)


## 18. Article-maintenance prompt and sequential runner

The maintainer receives both the new classified evidence and a mechanically derived summary of cumulative evidence coverage by category. The prompt makes the four statuses explicit, requires affirmative permissive evidence for `low_risk`, and prohibits converting missing evidence into reassurance. A validation pass checks each proposed status and permits one correction attempt when the output violates these semantics.


In [ ]:
def cumulative_category_coverage(evidence_ids, classification_by_id):
    """Summarize supported cumulative evidence by category and polarity."""
    coverage = {
        category: {
            "supported_restrictive_evidence_ids": [],
            "supported_permissive_evidence_ids": [],
            "supported_contextual_evidence_ids": [],
            "insufficiently_grounded_evidence_ids": [],
            "human_review_evidence_ids": [],
        }
        for category in THREAT_TAXONOMY
    }

    for evidence_id in evidence_ids:
        row = classification_by_id.get(evidence_id)
        if not row or not row.get("relevant"):
            continue

        grounding = row.get("grounding_status")
        for assignment in row.get("assignments", []):
            category = assignment.get("category")
            polarity = assignment.get("polarity")
            if category not in coverage or polarity not in POLARITIES:
                continue

            if row.get("requires_human_review"):
                coverage[category]["human_review_evidence_ids"].append(evidence_id)

            if grounding == "supported":
                key = f"supported_{polarity}_evidence_ids"
                coverage[category][key].append(evidence_id)
            else:
                coverage[category]["insufficiently_grounded_evidence_ids"].append(
                    evidence_id
                )

    for category_data in coverage.values():
        for key, values in list(category_data.items()):
            category_data[key] = sorted(set(values))
        category_data["has_supported_status_evidence"] = bool(
            category_data["supported_restrictive_evidence_ids"]
            or category_data["supported_permissive_evidence_ids"]
        )
        category_data["has_affirmative_low_risk_evidence"] = bool(
            category_data["supported_permissive_evidence_ids"]
        )

    return coverage


MAINTAINER_PROMPT = f"""You are the ARTICLE-MAINTENANCE stage of the
Maintainer in an NGO bow-tie monitoring system.

You receive:
- the five category articles at the prior revision;
- only the newly available evidence for the current revision;
- each record's filing/classification;
- cumulative_evidence_coverage, a mechanically derived summary of all supported
  evidence seen for each category up to the current cutoff.

Use only this input. Do not browse and do not add external facts.

Allowed article statuses and meanings:
{json.dumps(STATUS_DEFINITIONS, indent=2)}

Non-negotiable status semantics:
1. `unassessed` means evidence is absent or insufficient. A category with no
   supported restrictive or permissive evidence MUST be `unassessed`.
2. `low_risk` is an affirmative assessment, never a default. It requires at
   least one supported permissive evidence_id in cumulative_evidence_coverage
   that directly supports protection, reversal, non-applicability, or the
   current absence of a relevant restriction. No retrieved evidence, silence,
   contextual evidence alone, or the mere absence of restrictive evidence can
   never justify `low_risk`.
3. `high_risk` and `materialised` require at least one supported restrictive
   evidence_id. `materialised` additionally requires evidence that the
   restriction, loss, or condition is currently in effect.
4. Conflicting or inadequate evidence may remain `unassessed` even when some
   records exist. Do not force a judgment merely because the category is
   populated.
5. `status_basis_evidence_ids` must list the supported evidence that directly
   justifies the chosen status. It must be empty for `unassessed`; include at
   least one supported permissive id for `low_risk`; and include at least one
   supported restrictive id for `high_risk` or `materialised`.

Maintenance rules:
6. Return all five category articles at every revision.
7. Preserve supported prior claims unless new evidence supersedes them.
8. Every factual claim must cite one or more evidence_ids available in the
   cumulative record for that category.
9. Contextual evidence may clarify a narrative or open item, but cannot by
   itself justify any assessed status.
10. Evidence with grounding_status=unsupported must not support a claim or
    status change.
11. Evidence marked for human review may be included only with explicit
    uncertainty and should not by itself justify a high-confidence transition.
    If a transition rests solely on human-review-flagged evidence, describe it
    as provisional and set human_review_required=true.
12. A category can stay unchanged. Do not manufacture a transition.
13. The model proposes a status; a human remains responsible for accepting it.
14. List barriers that a human should verify. Do not recommend operational
    action.
15. Keep each article very concise. 
16. Statuses must be able to move down as well as up. Lower a status when the
    evidence for the prior status no longer holds, for example because a
    restrictive measure is repealed, withdrawn, struck down, suspended, lapsed,
    or documented as abandoned or superseded.
17. Any lowering must cite the superseding evidence_ids and set
    human_review_required=true. Contextual evidence may support a lowering only
    together with affirmative evidence that the prior driver was reversed or no
    longer applies; contextual evidence alone is insufficient.
18. If an `unassessed` category receives no new supported restrictive or
    permissive evidence, it must remain `unassessed`.

Output ONLY:
{{
  "revision_id": "...",
  "revision_summary": {{
    "changed_categories": ["..."],
    "narrative": "brief summary with evidence_ids"
  }},
  "articles": [
    {{
      "category": "one of the five categories",
      "status": "unassessed|low_risk|high_risk|materialised",
      "status_basis_evidence_ids": ["ev_..."],
      "summary": "short cumulative account",
      "claims": [
        {{"text": "claim", "evidence_ids": ["ev_..."]}}
      ],
      "open_items": ["unresolved question"],
      "barriers_to_verify": ["allowed category-level barrier"],
      "change_from_previous": "what changed or 'no change'",
      "human_review_required": true
    }}
  ]
}}
"""


def initial_articles():
    return [
        {
            "category": category,
            "status": "unassessed",
            "status_basis_evidence_ids": [],
            "summary": "Insufficient evidence has been assessed for this line.",
            "claims": [],
            "open_items": ["Establish sufficient evidence coverage for this line."],
            "barriers_to_verify": [],
            "change_from_previous": "initial unassessed state",
            "human_review_required": True,
        }
        for category in THREAT_TAXONOMY
    ]


def joined_evidence_record(item, classification):
    return {
        "evidence": compact_evidence(item),
        "classification": classification,
    }


def validate_maintainer_revision(revision, coverage_by_category):
    """Validate article completeness, citation scope, and four-state semantics."""
    issues = []
    articles = revision.get("articles")
    if not isinstance(articles, list):
        return ["Output does not contain an articles list."]

    returned_categories = [article.get("category") for article in articles]
    expected_categories = set(THREAT_TAXONOMY)
    if len(returned_categories) != len(expected_categories):
        issues.append(
            f"Expected {len(expected_categories)} articles, got {len(returned_categories)}."
        )
    if set(returned_categories) != expected_categories:
        issues.append(
            "Article categories must exactly match: "
            + ", ".join(sorted(expected_categories))
        )
    if len(returned_categories) != len(set(returned_categories)):
        issues.append("Article categories must not be duplicated.")

    for article in articles:
        category = article.get("category")
        if category not in coverage_by_category:
            continue

        status = article.get("status")
        basis_ids = article.get("status_basis_evidence_ids", [])
        if status not in ALLOWED_STATUSES:
            issues.append(f"{category}: invalid status {status!r}.")
            continue
        if not isinstance(basis_ids, list):
            issues.append(f"{category}: status_basis_evidence_ids must be a list.")
            basis_ids = []

        coverage = coverage_by_category[category]
        restrictive = set(coverage["supported_restrictive_evidence_ids"])
        permissive = set(coverage["supported_permissive_evidence_ids"])
        contextual = set(coverage["supported_contextual_evidence_ids"])
        supported = restrictive | permissive | contextual
        basis = set(basis_ids)

        unknown_basis = basis - supported
        if unknown_basis:
            issues.append(
                f"{category}: unsupported or cross-category status basis ids: "
                f"{sorted(unknown_basis)}."
            )

        if not (restrictive or permissive) and status != "unassessed":
            issues.append(
                f"{category}: no supported restrictive or permissive evidence; "
                "status must be unassessed."
            )
        if status == "unassessed" and basis:
            issues.append(f"{category}: unassessed must have an empty status basis.")
        if status == "low_risk" and not (basis & permissive):
            issues.append(
                f"{category}: low_risk requires a supported permissive evidence id."
            )
        if status in {"high_risk", "materialised"} and not (basis & restrictive):
            issues.append(
                f"{category}: {status} requires a supported restrictive evidence id."
            )

        for claim_index, claim in enumerate(article.get("claims", [])):
            claim_ids = set(claim.get("evidence_ids", []))
            invalid_claim_ids = claim_ids - supported
            if invalid_claim_ids:
                issues.append(
                    f"{category} claim {claim_index}: unsupported or cross-category "
                    f"evidence ids {sorted(invalid_claim_ids)}."
                )

    return issues


def call_maintainer(client, model, payload):
    """Request and validate one revision, allowing a bounded correction attempt."""
    base_user_content = json.dumps(payload, ensure_ascii=False, indent=2)
    messages = [{"role": "user", "content": base_user_content}]

    for attempt in range(MAX_MAINTAINER_REPAIR_ATTEMPTS + 1):
        resp = client.messages.create(
            model=model,
            max_tokens=16000,
            system=MAINTAINER_PROMPT,
            messages=messages,
        )

        if resp.stop_reason == "max_tokens":
            raise RuntimeError(
                f"Maintainer output for {payload['revision_id']} hit max_tokens; "
                "raise max_tokens or tighten article length in the prompt."
            )

        text = "".join(
            getattr(block, "text", "")
            for block in resp.content
            if getattr(block, "type", None) == "text"
        ).strip()
        revision = parse_json_text(text)
        issues = validate_maintainer_revision(
            revision, payload["cumulative_evidence_coverage"]
        )
        if not issues:
            revision["_raw_text"] = text
            revision["_repair_attempts"] = attempt
            return revision

        if attempt >= MAX_MAINTAINER_REPAIR_ATTEMPTS:
            formatted = "\n - ".join(issues)
            raise RuntimeError(
                f"Maintainer output for {payload['revision_id']} violated the "
                f"four-state semantics after {attempt + 1} attempt(s):\n - {formatted}"
            )

        messages.extend([
            {"role": "assistant", "content": text},
            {
                "role": "user",
                "content": (
                    "The proposed JSON violates the required four-state semantics. "
                    "Correct the output using the same payload and no external facts. "
                    "Return only the complete corrected JSON. Validation issues:\n- "
                    + "\n- ".join(issues)
                ),
            },
        ])

    raise RuntimeError("Unreachable maintainer retry state.")


def run_article_revisions(evidence_items, classification_rows, model):
    client = Anthropic()
    classification_by_id = {
        row.get("evidence_id"): row for row in classification_rows
    }

    items_by_window = {window["revision_id"]: [] for window in REVISION_WINDOWS}
    for item in evidence_items:
        revision_id = evidence_revision_id(item)
        if revision_id is not None and item["evidence_id"] in classification_by_id:
            items_by_window[revision_id].append(item)

    if sum(len(values) for values in items_by_window.values()) == 0:
        raise RuntimeError(
            "No evidence joined to any revision window. Inspect the pre-flight "
            "diagnostic before running article maintenance."
        )

    prior_articles = initial_articles()
    cumulative_evidence_ids = []
    run = {
        "stage": "article_maintenance",
        "model": model,
        "timestamp": utc_now_iso(),
        "editorial_policy": STATUS_DEFINITIONS,
        "status_semantics": {
            "absence_of_evidence_maps_to": "unassessed",
            "low_risk_requires_supported_permissive_evidence": True,
            "high_or_materialised_requires_supported_restrictive_evidence": True,
        },
        "revisions": [],
    }

    for window in REVISION_WINDOWS:
        new_items = items_by_window[window["revision_id"]]
        cumulative_evidence_ids.extend(item["evidence_id"] for item in new_items)
        coverage = cumulative_category_coverage(
            cumulative_evidence_ids, classification_by_id
        )

        if not new_items:
            carried = [
                {
                    **article,
                    "change_from_previous": (
                        "No change: no new evidence in this window "
                        "(carried forward without a model call)."
                    ),
                }
                for article in prior_articles
            ]
            run["revisions"].append({
                "revision_id": window["revision_id"],
                "revision_summary": {
                    "changed_categories": [],
                    "narrative": (
                        "No new classified evidence fell in this window; "
                        "articles were carried forward deterministically."
                    ),
                },
                "articles": carried,
                "_new_evidence_ids": [],
                "_model_called": False,
                "_repair_attempts": 0,
                "_coverage_by_category": coverage,
            })
            prior_articles = carried
            print(
                f"{window['revision_id']}: 0 new items — carried forward, no API call"
            )
            continue

        joined = [
            joined_evidence_record(
                item, classification_by_id[item["evidence_id"]]
            )
            for item in new_items
        ]

        payload = {
            "revision_id": window["revision_id"],
            "label": window["label"],
            "cutoff": window["cutoff"],
            "prior_articles": prior_articles,
            "new_classified_evidence": joined,
            "cumulative_evidence_coverage": coverage,
        }

        revision = call_maintainer(client, model, payload)
        revision["_new_evidence_ids"] = [
            item["evidence_id"] for item in new_items
        ]
        revision["_model_called"] = True
        revision["_coverage_by_category"] = coverage

        run["revisions"].append(revision)
        prior_articles = revision["articles"]
        print(
            f"{window['revision_id']}: {len(new_items)} new items — "
            f"changed: {revision.get('revision_summary', {}).get('changed_categories', [])}; "
            f"repair attempts: {revision.get('_repair_attempts', 0)}"
        )

    return run


## 19. Run and inspect the article revisions


In [ ]:
article_revision_run = run_article_revisions(items, classifications, MODEL)

with ARTICLE_REVISIONS_OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(article_revision_run, f, indent=2, ensure_ascii=False)

print(f"\nArticle revisions saved to {ARTICLE_REVISIONS_OUT_PATH}")

In [ ]:
for revision in article_revision_run.get("revisions", []):
    print("\n" + "=" * 80)
    print(
        revision.get("revision_id"),
        "(model called)" if revision.get("_model_called") else "(carried forward)",
    )
    print("New evidence:", revision.get("_new_evidence_ids", []))
    print(revision.get("revision_summary", {}).get("narrative", ""))

    for article in revision.get("articles", []):
        print(
            f"  {article.get('category')}: {article.get('status')} | "
            f"basis={article.get('status_basis_evidence_ids', [])} | "
            f"{article.get('change_from_previous')}"
        )


## 20. Audit the distinction between `unassessed` and assessed statuses

The table below exposes the evidence basis for every status. In a valid run, `unassessed` has no status-basis IDs, `low_risk` has at least one supported permissive ID, and `high_risk` or `materialised` has at least one supported restrictive ID.


In [ ]:
audit_rows = []
for revision in article_revision_run.get("revisions", []):
    coverage_by_category = revision.get("_coverage_by_category", {})
    for article in revision.get("articles", []):
        category = article.get("category")
        coverage = coverage_by_category.get(category, {})
        audit_rows.append({
            "revision_id": revision.get("revision_id"),
            "category": category,
            "status": article.get("status"),
            "status_basis_evidence_ids": ", ".join(
                article.get("status_basis_evidence_ids", [])
            ),
            "supported_restrictive_ids": ", ".join(
                coverage.get("supported_restrictive_evidence_ids", [])
            ),
            "supported_permissive_ids": ", ".join(
                coverage.get("supported_permissive_evidence_ids", [])
            ),
            "supported_contextual_ids": ", ".join(
                coverage.get("supported_contextual_evidence_ids", [])
            ),
            "human_review_required": article.get("human_review_required"),
            "repair_attempts": revision.get("_repair_attempts", 0),
        })

status_audit_df = pd.DataFrame(audit_rows)
display(status_audit_df)


In [ ]:
import json
import re
from pathlib import Path
from typing import Any


# ------------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------------

CATEGORY_ORDER = [
    "governance",
    "formation",
    "operations",
    "resources",
    "reputational_and_political",
]

CATEGORY_LABELS = {
    "governance": "Governance",
    "formation": "Formation",
    "operations": "Operations",
    "resources": "Resources",
    "reputational_and_political": "Reputational & political",
}

CATEGORY_COLORS = {
    "governance": {
        "fill": "#E5EDF6",
        "border": "#7594B5",
        "text": "#274B6D",
    },
    "formation": {
        "fill": "#F8EDCC",
        "border": "#D3A637",
        "text": "#80621B",
    },
    "operations": {
        "fill": "#DDF1EA",
        "border": "#59AD93",
        "text": "#26735E",
    },
    "resources": {
        "fill": "#ECE5F3",
        "border": "#9A7CB2",
        "text": "#654A7A",
    },
    "reputational_and_political": {
        "fill": "#F6DFD8",
        "border": "#D48970",
        "text": "#954C37",
    },
}

STATUS_COLORS = {
    "unassessed": {
        "label": "Unassessed",
        "fill": "#E7E7E7",
        "border": "#929292",
        "text": "#565656",
    },
    "low_risk": {
        "label": "Low risk",
        "fill": "#DCEFD9",
        "border": "#5BA467",
        "text": "#357A42",
    },
    "high_risk": {
        "label": "High risk",
        "fill": "#F6DA91",
        "border": "#C99016",
        "text": "#8A630C",
    },
    "materialised": {
        "label": "Materialised",
        "fill": "#F3C5BA",
        "border": "#CB604C",
        "text": "#9B382A",
    },
}

# Adjust these values to match your actual revision IDs.
REVISION_METADATA = {
    "baseline": {
        "title": "Baseline",
        "date": "31 DEC 2024",
        "order": 1,
    },
    "peak": {
        "title": "Peak",
        "date": "LATE 2025",
        "order": 2,
    },
    "divergence": {
        "title": "Divergence",
        "date": "JUNE 2026",
        "order": 3,
    },
}


# ------------------------------------------------------------------
# 2. Normalisation helpers
# ------------------------------------------------------------------

def normalise_identifier(value: Any) -> str:
    """Convert labels such as 'Reputational & Political' to snake_case."""
    text = str(value or "").strip().lower()
    text = text.replace("&", "and")
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def normalise_category(value: Any) -> str:
    category = normalise_identifier(value)

    aliases = {
        "reputational": "reputational_and_political",
        "reputational_political": "reputational_and_political",
        "reputational_and_political": "reputational_and_political",
        "reputation_and_political": "reputational_and_political",
    }

    return aliases.get(category, category)


def normalise_status(value: Any) -> str:
    status = normalise_identifier(value)

    aliases = {
        "low": "low_risk",
        "lowrisk": "low_risk",
        "low_risk": "low_risk",
        "high": "high_risk",
        "highrisk": "high_risk",
        "high_risk": "high_risk",
        "materialized": "materialised",
        "materialised": "materialised",
        "insufficient_evidence": "unassessed",
        "unassessed_insufficient_evidence": "unassessed",
        "unknown": "unassessed",
        "not_assessed": "unassessed",
    }

    status = aliases.get(status, status)

    if status not in STATUS_COLORS:
        return "unassessed"

    return status


def first_nonempty(mapping: dict, fields: list[str]) -> str:
    """Return the first usable textual field from an article."""
    for field in fields:
        value = mapping.get(field)

        if isinstance(value, str) and value.strip():
            return value.strip()

        if isinstance(value, list) and value:
            readable = [str(item).strip() for item in value if str(item).strip()]
            if readable:
                return "; ".join(readable)

    return ""


def shorten(text: str, max_chars: int = 85) -> str:
    """Create a compact caption suitable for a visual card."""
    clean = re.sub(r"\s+", " ", str(text or "")).strip()

    if not clean:
        return ""

    if len(clean) <= max_chars:
        return clean

    shortened = clean[: max_chars - 1].rsplit(" ", 1)[0]
    return shortened + "…"


def infer_revision_key(revision: dict, index: int) -> str:
    """
    Resolve varying revision IDs to baseline, peak, or divergence.
    Modify this if your notebook uses different revision names.
    """
    raw = normalise_identifier(
        revision.get("revision_id")
        or revision.get("name")
        or revision.get("label")
        or ""
    )

    if "baseline" in raw or "2024" in raw:
        return "baseline"

    if "peak" in raw or "2025" in raw:
        return "peak"

    if "divergence" in raw or "2026" in raw:
        return "divergence"

    fallback = ["baseline", "peak", "divergence"]
    return fallback[index] if index < len(fallback) else f"revision_{index + 1}"


# ------------------------------------------------------------------
# 3. Extract a short description for each visual card
# ------------------------------------------------------------------

def make_card_description(
    article: dict,
    coverage: dict,
    status: str,
) -> str:
    """
    Prefer a human-readable assessment field.

    Add or remove candidate fields depending on the exact article schema
    generated by your notebook.
    """
    candidate = first_nonempty(
        article,
        [
            "visual_summary",
            "short_summary",
            "status_summary",
            "status_rationale",
            "rationale",
            "assessment",
            "summary",
            "narrative",
            "current_assessment",
            "open_items",
        ],
    )

    if candidate:
        return shorten(candidate)

    permissive_ids = coverage.get(
        "supported_permissive_evidence_ids", []
    ) or []
    restrictive_ids = coverage.get(
        "supported_restrictive_evidence_ids", []
    ) or []
    contextual_ids = coverage.get(
        "supported_contextual_evidence_ids", []
    ) or []

    if status == "unassessed":
        return "Insufficient evidence coverage"

    if status == "low_risk" and permissive_ids:
        return f"Supported by permissive evidence: {', '.join(permissive_ids)}"

    if status in {"high_risk", "materialised"} and restrictive_ids:
        return f"Supported by restrictive evidence: {', '.join(restrictive_ids)}"

    if contextual_ids:
        return f"Contextual evidence: {', '.join(contextual_ids)}"

    return "See supporting evidence"


# ------------------------------------------------------------------
# 4. Build the visualisation JSON from article_revision_run
# ------------------------------------------------------------------

def build_bowtie_timeline_spec(article_revision_run: dict) -> dict:
    revisions_output = []

    revisions = article_revision_run.get("revisions", [])

    for revision_index, revision in enumerate(revisions):
        revision_key = infer_revision_key(revision, revision_index)

        metadata = REVISION_METADATA.get(
            revision_key,
            {
                "title": revision.get(
                    "label",
                    revision.get("revision_id", revision_key),
                ),
                "date": str(revision.get("cutoff_date", "")),
                "order": revision_index + 1,
            },
        )

        coverage_by_category = revision.get(
            "_coverage_by_category", {}
        ) or {}

        articles_by_category = {
            normalise_category(article.get("category")): article
            for article in revision.get("articles", [])
        }

        cards = []

        for category in CATEGORY_ORDER:
            article = articles_by_category.get(category, {})
            status = normalise_status(article.get("status"))

            coverage = (
                coverage_by_category.get(category)
                or coverage_by_category.get(
                    CATEGORY_LABELS.get(category, category)
                )
                or {}
            )

            basis_ids = article.get(
                "status_basis_evidence_ids", []
            ) or []

            cards.append(
                {
                    "category_id": category,
                    "category_label": CATEGORY_LABELS[category],
                    "category_style": CATEGORY_COLORS[category],
                    "description": make_card_description(
                        article=article,
                        coverage=coverage,
                        status=status,
                    ),
                    "status": status,
                    "status_label": STATUS_COLORS[status]["label"],
                    "status_style": STATUS_COLORS[status],
                    "status_basis_evidence_ids": basis_ids,
                    "human_review_required": bool(
                        article.get("human_review_required", False)
                    ),
                    "evidence_coverage": {
                        "restrictive": coverage.get(
                            "supported_restrictive_evidence_ids", []
                        )
                        or [],
                        "permissive": coverage.get(
                            "supported_permissive_evidence_ids", []
                        )
                        or [],
                        "contextual": coverage.get(
                            "supported_contextual_evidence_ids", []
                        )
                        or [],
                    },
                }
            )

        revisions_output.append(
            {
                "revision_id": revision.get(
                    "revision_id", revision_key
                ),
                "revision_key": revision_key,
                "title": metadata["title"],
                "date": metadata["date"],
                "order": metadata["order"],
                "repair_attempts": revision.get(
                    "_repair_attempts", 0
                ),
                "cards": cards,
            }
        )

    revisions_output.sort(key=lambda revision: revision["order"])

    return {
        "visualisation_type": "bowtie_risk_timeline",
        "title": "Nepal NGO Civic-Space Risk Trajectory",
        "subtitle": (
            "Threat-line status across three retrospective assessment windows"
        ),
        "top_event": {
            "label": "Loss of operational viability in a country",
            "description": (
                "The accumulated restrictiveness of the operating "
                "environment exceeds what the NGO requires to deliver "
                "its core mandate."
            ),
        },
        "layout": {
            "orientation": "landscape",
            "revision_columns": len(revisions_output),
            "category_rows": len(CATEGORY_ORDER),
            "card_structure": (
                "category card, right-pointing arrow, status card"
            ),
            "show_evidence_ids": False,
            "show_review_marker": True,
            "review_marker": "small warning icon",
            "show_legend": True,
            "minimal_background": True,
        },
        "legend": [
            {
                "status": status,
                **style,
            }
            for status, style in STATUS_COLORS.items()
        ],
        "revisions": revisions_output,
    }


visualisation_spec = build_bowtie_timeline_spec(
    article_revision_run
)

display(visualisation_spec)

In [ ]:
OUTPUT_JSON = Path("nepal_bowtie_timeline_v4.json")

with OUTPUT_JSON.open("w", encoding="utf-8") as file:
    json.dump(
        visualisation_spec,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Visualisation JSON saved to: {OUTPUT_JSON.resolve()}")

In [ ]:
#!/usr/bin/env python3
"""Render a bowtie_risk_timeline JSON spec as a standalone SVG via the Claude API.

Usage:
    Set ANTHROPIC_API_KEY in a local .env file (see .env.example), or
    export ANTHROPIC_API_KEY=... directly, then:
        python render_bowtie_svg.py

Requires: pip install anthropic python-dotenv  (and optionally cairosvg for PDF output)
"""

import base64
import json
import pathlib
import re

import anthropic
from dotenv import load_dotenv

MODEL = "claude-sonnet-5"
SPEC_PATH = pathlib.Path("nepal_bowtie_timeline_v4.json")   # your JSON spec
REFERENCE_PDF = None#pathlib.Path("example_nepal.pdf")        # style reference; set to None to skip
OUT_SVG = pathlib.Path("nepal_bowtie_timeline.svg")
OUT_PDF = pathlib.Path("nepal_bowtie_timeline.pdf")      # for \includegraphics; needs cairosvg

PROMPT = """You are rendering a figure for an academic paper. The JSON below is a \
`bowtie_risk_timeline` specification. Output ONE complete, standalone SVG document and \
nothing else: no markdown fences, no commentary, no XML prolog.

Layout rules:
- Landscape, viewBox="0 0 1560 780", white background rect, generic sans-serif font stack \
(Helvetica, Arial, sans-serif). No external fonts, images, scripts, or CSS imports.
- Render the revision columns left to right in `revisions[].order`. Column header: `title` \
in bold ~20px centred, with `date` beneath in ~12px grey uppercase with letter-spacing.
- Within each column, render the five cards top to bottom in the given order. Each row is: \
a category card (rounded rect ~230x80, fill/stroke/text colours taken exactly from \
`category_style`; bold `category_label` on the first line, `description` beneath in ~11px, \
broken into at most two <tspan> lines and cut with an ellipsis if longer) -> a short grey \
right-pointing arrow -> a status card (rounded rect ~150x80, colours exactly from \
`status_style`, bold `status_label` centred).
- Rows must align horizontally across the three columns so the figure reads as a timeline.
- If `human_review_required` is true, draw a small warning marker at the top-right corner \
of the status card: a tiny triangle with an exclamation mark, built from SVG shapes (no emoji).
- Do not display evidence IDs anywhere (`show_evidence_ids` is false).
- Legend centred at the bottom, built from the `legend` array in order: a bold "Status:" \
label, then for each entry a small rounded swatch (entry fill + border) followed by its label.
- Use only the hex colours provided in the JSON; do not invent colours.

If a reference document is attached, match its visual style (spacing, corner radii, \
typography, arrow style, overall proportions) as closely as possible, but take all data, \
labels, statuses, and legend entries from the JSON, not from the reference.

JSON specification:
"""


def main() -> None:
    load_dotenv()  # reads ANTHROPIC_API_KEY from a local .env file, if present
    client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from the environment

    spec = json.loads(SPEC_PATH.read_text(encoding="utf-8"))

    content: list[dict] = []
    if REFERENCE_PDF and REFERENCE_PDF.exists():
        content.append(
            {
                "type": "document",
                "source": {
                    "type": "base64",
                    "media_type": "application/pdf",
                    "data": base64.standard_b64encode(REFERENCE_PDF.read_bytes()).decode(),
                },
            }
        )
    content.append(
        {
            "type": "text",
            "text": PROMPT + json.dumps(spec, ensure_ascii=False, indent=2),
        }
    )

    # Note for claude-sonnet-5: do NOT pass temperature/top_p/top_k (non-default
    # values return a 400) and do not set a manual `thinking` block; adaptive
    # thinking is on by default. max_tokens is generous because a 3x5-card SVG
    # is long, and Sonnet 5's tokenizer counts ~30% more tokens than 4.6.
    with client.messages.stream(
        model=MODEL,
        max_tokens=32000,
        messages=[{"role": "user", "content": content}],
    ) as stream:
        message = stream.get_final_message()

    text = "".join(block.text for block in message.content if block.type == "text")
    match = re.search(r"<svg\b.*</svg>", text, re.DOTALL)
    if not match:
        raise RuntimeError(f"No SVG found in response:\n{text[:500]}")

    OUT_SVG.write_text(match.group(0), encoding="utf-8")
    print(f"Wrote {OUT_SVG} ({OUT_SVG.stat().st_size} bytes)")

    # Optional: convert to PDF for LaTeX inclusion (pip install cairosvg)
    try:
        import cairosvg

        cairosvg.svg2pdf(url=str(OUT_SVG), write_to=str(OUT_PDF))
        print(f"Wrote {OUT_PDF}")
    except ImportError:
        print("cairosvg not installed; skipping PDF conversion")


if __name__ == "__main__":
    main()

## 21. What this pilot supports

After manual checking, the notebook can support a bounded claim such as:

> We instantiated collection, filing, and sequential article maintenance for the Nepal case. The Collector produced provenance-bearing candidate evidence; a separate filing stage mapped each record to the bow-tie taxonomy, polarity, and affected barrier; and the article-maintenance stage proposed citation-bound revisions under a four-state policy. The policy distinguishes insufficient evidence (`unassessed`) from an affirmative low-risk assessment and requires supported permissive evidence before assigning `low_risk`.

The notebook does not establish production-level source recall, continuous change detection, cross-language performance, status calibration, or expert-level classification accuracy. Those claims require a gold set and human evaluation. Proposed status changes, especially de-escalations and transitions based on review-flagged evidence, remain subject to human acceptance.
